In [9]:
import os

import torch
from ours import Model
import pandas as pd
import numpy as np

DATASET_REGISTRY = {
    'PJM': {
        'enc_in': 24,  # 19 subs + 5 weather
        'subs': ['AE','AEP','AP','ATSI','BC','CE','DAY','DEOK','DOM',
                 'DPL','DUQ','EKPC','JC','ME','PE','PEP','PL','PN','PS'],
    },
    'NYIS': {
        'enc_in': 15,  # 10 + 5
        'subs': ['ZONA','ZONB','ZONC','ZOND','ZONE','ZONF','ZONG','ZONH','ZONI','ZONJ'],
    },
    'ISNE': {
        'enc_in': 13,  # 8 + 5
        'subs': ['4001','4002','4003','4004','4005','4006','4007','4008'],
    },
}

# 1. Configure
class Cfg:
    seq_len = 168;  pred_len = 24;  enc_in = 1
    time_emb_dim = 4;  number_of_targets = 0
    F_conv_output = 10;  F_noconv_output = 10
    F_gconv_output_rate = 1;  F_lin_output = 10
    F_id_output_rate = 1;  F_emb_output = 10
    mlp_ratio = 1;  predictor = 'Default';  predictor_dropout = 0.1
    temporal_gate = False;  channel_sparsity = False
    label_len = 168;  single_tmpf = False
    # add any missing attrs your predictor needs
    dropout = 0.1;  output_attention = False
    continuity_beta = 1
    d_model = 128;  n_heads = 8;  d_ff = 64;  activation = 'gelu'


def load_data(dataset, sub_name, root_path='./'):
    """Load OT + weather, join, return raw numpy arrays."""
    info = DATASET_REGISTRY[dataset]

    # Load OT
    df = pd.read_csv(os.path.join(root_path, f'{dataset}_OT_data.csv'), index_col='date')
    df = df[info['subs']]
    
    # Build new_cols: target sub first, then the rest (same as run.py)
    feature_list = info['subs']
    sub_index = feature_list.index(sub_name)
    new_cols = [sub_name] + [s for j, s in enumerate(feature_list) if j != sub_index]
    df = df[new_cols]

    # Load weather for this sub, join
    df_w = pd.read_csv(os.path.join(root_path, f'{dataset}_weather_data.csv'), index_col='date')
    df_w = df_w[df_w['sub'].astype(str) == str(sub_name)]
    df_w = df_w[['tmpf', 'relh', 'sknt', 'alti', 'vsby']]
    df = df.join(df_w, how='left')

    # Test: 2024, plus seq_len lookback
    test = df[(df.index >= '2024-01-01') & (df.index < '2025-01-01')]
    start = df.index.get_loc(test.index[0]) - 168
    data = df.values[start:]
    dates = pd.to_datetime(df.index[start:start + len(data)])

    # timeenc=0: month, day, weekday, hour
    dates = pd.to_datetime(df.index[start:start + len(data)])
    stamps = np.stack([
        dates.month,
        dates.day,
        dates.weekday,
        dates.hour,
    ], axis=-1).astype(np.float32)  # (n_hours, 4)

    return data, stamps


def build_one_sample(data, stamps, idx=0):
    """
    Build the 4 model inputs: x_enc, x_mark_enc, x_dec, x_mark_dec
    
    x_enc      = (batch_x, batch_weather)   tuple
    x_mark_enc = batch_x_mark               (1, 168, 4)
    x_dec      = dec_inp                     (1, 192, enc_in)
    x_mark_dec = batch_y_mark                (1, 192, 4)
    """
    seq_len, pred_len, label_len = 168, 24, 168

    s = idx
    e = s + seq_len

    # encoder input
    batch_x = torch.tensor(data[s:e], dtype=torch.float32).unsqueeze(0)          # (1, 168, enc_in)
    x_enc = batch_x

    # encoder time marks
    x_mark_enc = torch.tensor(stamps[s:e], dtype=torch.float32).unsqueeze(0)     # (1, 168, 4)

    # ground truth for decoder construction
    batch_y = torch.tensor(data[e - label_len:e + pred_len], dtype=torch.float32).unsqueeze(0)  # (1, 192, enc_in)

    # decoder time marks
    x_mark_dec = torch.tensor(stamps[e - label_len:e + pred_len], dtype=torch.float32).unsqueeze(0)  # (1, 192, 4)

    # true target
    true = batch_y[:, -pred_len:, [0]]  # (1, 24, 1)

    return x_enc, x_mark_enc, x_mark_dec, true



def build_model(dataset, checkpoint_path, device='cpu'):
    """
    Build CATS model for sub_train + weather_train mode.
    enc_in = n_subs + weather_enc
    """
    info = DATASET_REGISTRY[dataset]
    cfg = Cfg()
    cfg.enc_in = info['enc_in']# e.g. PJM: 19+5=24
    cfg.single_tmpf = False

    model = Model(cfg).float().to(device)
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.eval()
    return model, cfg


def inference(model, cfg, x_enc, x_mark_enc, x_mark_dec, device='cpu'):
    """
    x_enc:     (B, seq_len, n_subs + weather_enc)  — load + weather already concatenated
    x_weather: (B, seq_len, weather_dim)            — passed in tuple but unused when single_tmpf=False
    """
    with torch.no_grad():
        x_enc = x_enc.float().to(device)
        x_mark = x_mark_enc.float().to(device)
        y_mark = x_mark_dec.float().to(device)

        batch_x = x_enc

        dec_inp = torch.zeros(x_enc.shape[0], cfg.label_len + cfg.pred_len,
                              x_enc.shape[-1]).float().to(device)

        output = model((batch_x, _), x_mark, dec_inp, y_mark)
        pred = output[:, -cfg.pred_len:, [0]]
    return pred


# ── PJM: enc_in = 19 subs + 5 weather = 24 ──
dataset = 'PJM';  sub = 'AE'
model, cfg = build_model(dataset, 'ours_' + sub + '_MS_best_model.pth')
data, stamps = load_data(dataset, sub)
x_enc, x_mark_enc, x_mark_dec, true = build_one_sample(data, stamps, idx=0)

pred = inference(
    model, cfg,
    x_enc, x_mark_enc, x_mark_dec
)

print(pred)
print(f"Dataset: {dataset} Sub: {sub}  enc_in={cfg.enc_in}  output: {pred.shape}")  # (1, 24, 1)
print(f"True target: {true.squeeze().numpy()}")




Current Predictor: Default


/tmp/ipykernel_810396/1151811963.py:123: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(checkpoint_path, map_location=device))


tensor([[[1087.3031],
         [1033.8300],
         [ 995.0344],
         [ 974.3167],
         [ 977.3589],
         [ 979.0108],
         [ 994.1069],
         [1024.7015],
         [1049.5659],
         [1044.4562],
         [1010.2272],
         [ 996.5921],
         [ 995.9515],
         [1031.3464],
         [1038.7347],
         [1074.2095],
         [1133.6957],
         [1214.2805],
         [1281.2572],
         [1295.4131],
         [1265.0660],
         [1229.6246],
         [1189.4749],
         [1129.2924]]])
Dataset: PJM Sub: AE  enc_in=24  output: torch.Size([1, 24, 1])
True target: [1103. 1055. 1016.  986.  966.  962.  974.  994. 1004. 1002.  964.  927.
  935.  975. 1019. 1072. 1132. 1197. 1278. 1271. 1252. 1221. 1173. 1103.]
